# QMC-HAMM Data Repository Demo
This notebook shows how to query configuration files from the
public repository.

We will install our library from GitHub repo.

In [4]:
!pip install git+https://github.com/qmc-hamm/configurations.git
!pip install rich

  Cloning https://github.com/qmc-hamm/configurations.git to /tmp/pip-req-build-hetkte3s
  Running command git clone --filter=blob:none --quiet https://github.com/qmc-hamm/configurations.git /tmp/pip-req-build-hetkte3s
  Resolved https://github.com/qmc-hamm/configurations.git to commit 39ce2689e081866370c8f63594efd77e8af193fc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached rich-14.3.2-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached rich-14.3.2-py3-none-any.whl (309 kB)
Using cached markdown_it_py-4.0.0-py3-none-any.whl (87 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [rich]2/3 [rich]


In [11]:
from qmc_repo.repo import Repo
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
import h5py

console = Console()

In [6]:
# Connect to the repository of QMC Data
repo = Repo()

In [7]:
# List all of the catalog entries
table = Table(title="Catalog Entries", show_lines=True)
table.add_column("Name", style="cyan", justify="right")
table.add_column("Description", style="magenta", justify="left")
table.add_column("Version", style="yellow", justify="right")
for entry in repo.entries:
    source = repo.get_source(entry)
    table.add_row(entry, source.description, source.metadata.get("version", "-"))
console.print(table)


                 Catalog Entries                  
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃        Name ┃ Description            ┃ Version ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ hydrogen_v1 │ QMC HAMM Hydrogen Data │     1.0 │
└─────────────┴────────────────────────┴─────────┘

In [8]:
# Use version 1.0 of the hydrogen source
source = repo.hydrogen_v1

In [9]:
# Display source info in a panel
source_info = (
    f"[bold]Description:[/bold] {source.description}\n"
    f"[bold]Container:[/bold] {source.container}\n"
    f"[bold]Version:[/bold] {source.metadata.get('version', 'unknown')}"
)
console.print(Panel(source_info, title="[bold cyan]Source: hydrogen[/bold cyan]", expand=False))
console.print()


╭───────── Source: hydrogen ──────────╮
│ Description: QMC HAMM Hydrogen Data │
│ Container: dataframe                │
│ Version: 1.0                        │
╰─────────────────────────────────────╯

In [12]:
# Load the data
df = source.read()

# Filter for pressure == 140 and temperature == 2400
high_pressure = df[(df["pressure"] == 140) & (df["temperature"] == 2400)]

console.print(f"[bold]Entries with pressure == 140 and temperature == 2400:[/bold]")
console.print(f"Found [green]{len(high_pressure)}[/green] entries out of [blue]{len(df)}[/blue] total")
console.print()

# Create a table for HDF5 attributes with selected interesting fields
table = Table(title="HDF5 File Attributes", expand=True, show_lines=True)
table.add_column("Config", style="cyan", justify="right")
table.add_column("P (GPa)", style="magenta", justify="right")
table.add_column("T (K)", style="magenta", justify="right")
table.add_column("State", style="yellow")
table.add_column("rs", style="green", justify="right")
table.add_column("Mol %", style="green", justify="right")
table.add_column("Method", style="blue")
table.add_column("Model", style="blue")
table.add_column("Potential Energy (eV)", style="red", justify="right")
table.add_column("Datasets", style="dim")

for row in high_pressure.itertuples():
    with repo.fs.open(row.uri, 'rb') as f:
        with h5py.File(f, 'r') as h5f:
            attrs = dict(h5f.attrs)
            datasets = list(h5f.keys())

            table.add_row(
                str(attrs.get("config_number", "-")),
                str(attrs.get("pressure", "-")),
                str(attrs.get("temperature", "-")),
                str(attrs.get("state", "-")),
                f"{attrs.get('rs', '-'):.2f}" if attrs.get("rs") else "-",
                f"{attrs.get('molecular_percentage', '-'):.1f}" if attrs.get("molecular_percentage") else "-",
                str(attrs.get("method", "-")),
                str(attrs.get("modelname", "-")),
                f"{attrs.get('potential_energy', '-'):.4f}" if attrs.get("potential_energy") else "-",
                ", ".join(datasets),
            )

console.print(table)

Entries with pressure == 140 and temperature == 2400:

Found 4 entries out of 95 total

                                               HDF5 File Attributes                                                
┏━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃        ┃         ┃       ┃        ┃      ┃       ┃        ┃       ┃     Potential Energy ┃                      ┃
┃ Config ┃ P (GPa) ┃ T (K) ┃ State  ┃   rs ┃ Mol % ┃ Method ┃ Model ┃                 (eV) ┃ Datasets             ┃
┡━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│     81 │     140 │  2400 │ liquid │ 1.52 │   0.6 │ NPT    │ M18   │           -3289.8374 │ electronic_sk_data,  │
│        │         │       │        │      │       │        │       │                      │ gofr_data,           │
│        │         │       │        │      │       │        │       │                      │ sofk_data, xyz_data  │
├────────┼─────────┼───────┼────────┼──────┼───────┼────────┼───────┼──────────────────────┼──────────────────────┤
│      9 │     140 │  2400 │ liquid │ 1.53 │   0.5 │ NPT    │ M18   │                    - │ sofk_data, xyz_data  │
├────────┼─────────┼───────┼────────┼──────┼───────┼────────┼───────┼──────────────────────┼──────────────────────┤
│     90 │     140 │  2400 │ liquid │ 1.52 │   0.4 │ NPT    │ M18   │           -3293.5641 │ electronic_sk_data,  │
│        │         │       │        │      │       │        │       │                      │ gofr_data,           │
│        │         │       │        │      │       │        │       │                      │ sofk_data, xyz_data  │
├────────┼─────────┼───────┼────────┼──────┼───────┼────────┼───────┼──────────────────────┼──────────────────────┤
│     99 │     140 │  2400 │ liquid │ 1.51 │   0.5 │ NPT    │ M18   │           -3275.8980 │ electronic_sk_data,  │
│        │         │       │        │      │       │        │       │                      │ gofr_data,           │
│        │         │       │        │      │       │        │       │                      │ sofk_data, xyz_data  │
└────────┴─────────┴───────┴────────┴──────┴───────┴────────┴───────┴──────────────────────┴──────────────────────┘